In [2]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [3]:
import os
import chromadb
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

CHROMA_PATH = "vector_store"
EMBEDDING_MODEL = "text-embedding-3-large"
GENERATION_MODEL = "gpt-5.4-mini"

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = chroma_client.get_or_create_collection(
    name="technical_docs"
)

print("ChromaDB collection loaded.")
print("Stored records:", collection.count())

ChromaDB collection loaded.
Stored records: 37


In [4]:
QUESTION = input(
    "Enter your question: "
).strip()

CATEGORY_FILTER = input(
    "Enter category filter (press Enter to search all documents): "
).strip()

if not CATEGORY_FILTER:
    CATEGORY_FILTER = None

TOP_K = 3

print("\nQuestion:", QUESTION)
print("Top K:", TOP_K)
print(
    "Category Filter:",
    CATEGORY_FILTER if CATEGORY_FILTER else "None - searching all documents"
)


Question: what is databricks?
Top K: 3
Category Filter: None - searching all documents


In [5]:
question_response = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=[QUESTION]
)

question_embedding = question_response.data[0].embedding

print("Question embedding generated.")
print(
    "Embedding dimension:",
    len(question_embedding)
)

Question embedding generated.
Embedding dimension: 3072


In [6]:
query_args = {
    "query_embeddings": [question_embedding],
    "n_results": TOP_K
}

if CATEGORY_FILTER:
    query_args["where"] = {
        "category": CATEGORY_FILTER
    }

results = collection.query(
    **query_args
)

print(f"Top {TOP_K} retrieved chunks:\n")

for i in range(len(results["ids"][0])):
    metadata = results["metadatas"][0][i]

    print(f"Result {i + 1}")
    print("Title:", metadata["title"])
    print("Category:", metadata["category"])
    print("Source:", metadata["source_file"])
    print("Chunk ID:", results["ids"][0][i])
    print(
        "Distance:",
        round(results["distances"][0][i], 4)
    )

    print("\nText:")
    print(
        results["documents"][0][i][:250]
    )

    print("-" * 60)

Top 3 retrieved chunks:

Result 1
Title: Azure Databricks
Category: Azure Databricks
Source: azure_databricks.pdf
Chunk ID: DOC5_chunk_1
Distance: 0.7209

Text:
Azure Databricks

Databricks and the Lakehouse
Azure Databricks is a managed analytics platform built around Apache Spark and lakehouse concepts.
It supports notebooks, SQL, jobs, workflows, governance, and Delta Lake. Data engineering workloads
comm
------------------------------------------------------------
Result 2
Title: Azure Databricks
Category: Azure Databricks
Source: azure_databricks.pdf
Chunk ID: DOC5_chunk_2
Distance: 1.0708

Text:
transactions, schema enforcement and evolution, updates, deletes, merges, and versioned table
history. MERGE is especially useful for incremental pipelines that need to insert new records and
update matching records.
Practical note. For data engineer
------------------------------------------------------------
Result 3
Title: Azure Databricks
Category: Azure Databricks
Source: azure_datab

In [7]:
retrieved_context = "\n\n".join(
    results["documents"][0]
)

prompt = f"""
Answer the question using only the retrieved context below.

If the retrieved context does not contain enough information,
say that the available documents do not provide enough information.

Question:
{QUESTION}

Retrieved Context:
{retrieved_context}
"""

response = client.responses.create(
    model=GENERATION_MODEL,
    input=prompt
)

generated_answer = response.output_text

print("Question:")
print(QUESTION)

if CATEGORY_FILTER:
    print("\nCategory Filter:")
    print(CATEGORY_FILTER)

print("\nGenerated Answer:")
print(generated_answer)

Question:
what is databricks?

Generated Answer:
Databricks is described in the retrieved context as a managed analytics platform built around Apache Spark and lakehouse concepts. It supports notebooks, SQL, jobs, workflows, governance, and Delta Lake.


In [8]:
print("\nSources Used:")

for i, metadata in enumerate(
    results["metadatas"][0],
    start=1
):
    print(
    f"{i}. "
    f"{metadata['title']} | "
    f"{metadata['source_file']} | "
    f"{results['ids'][0][i - 1]} | "
    f"Batch: {metadata['batch_id']}"
)


Sources Used:
1. Azure Databricks | azure_databricks.pdf | DOC5_chunk_1 | Batch: batch_2
2. Azure Databricks | azure_databricks.pdf | DOC5_chunk_2 | Batch: batch_2
3. Azure Databricks | azure_databricks.pdf | DOC5_chunk_3 | Batch: batch_2
